# Agentic RAG Aviation Safety - Workflow Notebook

This notebook demonstrates the Agentic RAG workflow for aviation safety queries.

## Quick Start

1. Start the services: `docker-compose up -d`
2. Set your API key: `export ANTHROPIC_API_KEY=sk-ant-...` (or `OPENAI_API_KEY`)
3. Run this notebook

## Configuration

The following environment variables can be configured:

| Variable | Default | Description |
|----------|---------|-------------|
| `QDRANT_URL` | `http://localhost:6333` | Qdrant server URL |
| `LLM_PROVIDER` | `anthropic` | `openai` or `anthropic` |
| `ANTHROPIC_API_KEY` | (required) | Anthropic API key |
| `OPENAI_API_KEY` | (required if using OpenAI) | OpenAI API key |
| `MLFLOW_TRACKING_URI` | `http://localhost:5001` | MLflow tracking server |
| `MLFLOW_EXPERIMENT_NAME` | `agentic-rag-aviation` | Custom experiment name |

## 1. Setup and Configuration

In [ ]:
import os
import sys

# =============================================================================
# ENVIRONMENT CONFIGURATION
# =============================================================================
# Set defaults for local development - override via environment variables as needed

# Qdrant Vector Database URL
if not os.environ.get('QDRANT_URL'):
    os.environ['QDRANT_URL'] = 'http://localhost:6333'

# LLM Provider (anthropic or openai)
if not os.environ.get('LLM_PROVIDER'):
    os.environ['LLM_PROVIDER'] = 'anthropic'

# MLflow tracking URI
if not os.environ.get('MLFLOW_TRACKING_URI'):
    os.environ['MLFLOW_TRACKING_URI'] = 'http://localhost:5001'

# MLflow experiment name
if not os.environ.get('MLFLOW_EXPERIMENT_NAME'):
    os.environ['MLFLOW_EXPERIMENT_NAME'] = 'agentic-rag-aviation'

# =============================================================================
# PATH SETUP
# =============================================================================
# Add src to path if running from notebooks folder
src_path = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# =============================================================================
# VERIFY CONFIGURATION
# =============================================================================
print("=" * 60)
print("ENVIRONMENT CONFIGURATION")
print("=" * 60)

llm_provider = os.environ.get('LLM_PROVIDER', 'anthropic')
print(f"LLM_PROVIDER: {llm_provider}")

if llm_provider == 'anthropic':
    if not os.environ.get('ANTHROPIC_API_KEY'):
        print("ANTHROPIC_API_KEY: NOT SET - Required for Anthropic!")
    else:
        print("ANTHROPIC_API_KEY: Set")
else:
    if not os.environ.get('OPENAI_API_KEY'):
        print("OPENAI_API_KEY: NOT SET - Required for OpenAI!")
    else:
        print("OPENAI_API_KEY: Set")

print(f"QDRANT_URL: {os.environ.get('QDRANT_URL')}")
print(f"MLFLOW_TRACKING_URI: {os.environ.get('MLFLOW_TRACKING_URI')}")
print(f"MLFLOW_EXPERIMENT_NAME: {os.environ.get('MLFLOW_EXPERIMENT_NAME')}")
print("=" * 60)

## 2. Configure MLflow Tracing

MLflow is configured via environment variables. In Domino, `MLFLOW_TRACKING_URI` is automatically set.

You can customize the experiment name by setting `MLFLOW_EXPERIMENT_NAME` before importing the modules.

In [ ]:
import mlflow

# MLflow client is initialized from MLFLOW_TRACKING_URI environment variable
# In Domino, this is automatically configured - no parameters needed
client = mlflow.MlflowClient()

# Set custom experiment name (optional)
experiment_name = os.environ.get('MLFLOW_EXPERIMENT_NAME', 'agentic-rag-aviation')
mlflow.set_experiment(experiment_name)
print(f"MLflow Experiment: {experiment_name}")

# Verify connection
try:
    experiment = mlflow.get_experiment_by_name(experiment_name)
    if experiment:
        print(f"Experiment ID: {experiment.experiment_id}")
    else:
        print("Experiment will be created on first run")
except Exception as e:
    print(f"MLflow connection status: {e}")

## 3. Initialize the Agentic RAG Pipeline

In [ ]:
from agentic_rag.pipelines import AgenticRAG, TraditionalRAG
from agentic_rag.models import RefinementMode
from agentic_rag.config import get_settings

# Display current configuration
settings = get_settings()
print("Current Configuration:")
print(f"  LLM Provider: {settings.llm_provider}")
print(f"  Refinement Model: {settings.refinement_model}")
print(f"  Generation Model: {settings.generation_model}")
print(f"  Embedder: {settings.embedder}")
print(f"  Embedding Model: {settings.embedding_model}")
print(f"  MLflow Enabled: {settings.mlflow_enabled}")

# Initialize pipelines
agentic = AgenticRAG()
traditional = TraditionalRAG()
print("\nPipelines initialized successfully!")

## 4. Run a Query with Agentic RAG

In [ ]:
# Example query
question = "What are common causes of Cessna 172 accidents?"

# Run agentic RAG with tracing
response = await agentic.query(
    question=question,
    refinement_mode=RefinementMode.DEDUP,  # Options: DEDUP, SYNTHESIZE, NONE
    top_k=10,
    include_trace=True
)

print(f"Question: {question}\n")
print("=" * 60)
print("ANSWER SUMMARY:")
print("=" * 60)
print(response.answer.summary)

In [ ]:
# Display key findings
if response.answer.key_findings:
    print("\nKEY FINDINGS:")
    print("-" * 40)
    for finding in response.answer.key_findings:
        print(f"  - {finding.finding}")
        print(f"    Source: {finding.source}, Confidence: {finding.confidence}")
        print()

In [ ]:
# Display execution trace
if response.trace:
    print("\nEXECUTION TRACE:")
    print("-" * 40)
    print(f"Intent: {response.trace.intent} (confidence: {response.trace.intent_confidence})")
    print(f"Strategy: {response.trace.strategy}")
    print(f"Sources used: {response.trace.sources_used}")
    
    # Get document counts from refinement dict if available
    if response.trace.refinement:
        print(f"Documents before refinement: {response.trace.refinement.get('input_count', 'N/A')}")
        print(f"Documents after refinement: {response.trace.refinement.get('output_count', 'N/A')}")
        print(f"Refinement mode: {response.trace.refinement.get('mode', 'N/A')}")
    
    # Get retrieval info
    if response.trace.retrievals:
        total_docs = sum(r.get('document_count', 0) for r in response.trace.retrievals)
        print(f"Total documents retrieved: {total_docs}")
    
    print(f"Total duration: {response.trace.total_duration_ms:.0f}ms" if response.trace.total_duration_ms else "Duration: N/A")
    
    # Show MLflow links if available
    if response.trace.mlflow_run_id:
        print(f"\nMLflow Run ID: {response.trace.mlflow_run_id}")

## 5. Compare Agentic vs Traditional RAG

In [ ]:
# Run comparison
comparison_question = "What accidents have occurred due to fuel exhaustion?"

comparison = await agentic.compare(
    question=comparison_question,
    top_k=10
)

print(f"Question: {comparison_question}\n")
print("=" * 60)
print("TRADITIONAL RAG:")
print("=" * 60)
print(f"Documents used: {comparison['traditional']['documents_used']}")
print(f"Answer: {comparison['traditional']['answer'][:500]}...\n")

print("=" * 60)
print("AGENTIC RAG:")
print("=" * 60)
print(f"Answer: {comparison['agentic']['answer']['summary']}")

## 6. Direct Retrieval (Low-Level API)

In [ ]:
from agentic_rag.retrieval import IncidentRetriever, RegulationRetriever, NewsRetriever

# Initialize retrievers
incident_retriever = IncidentRetriever()
news_retriever = NewsRetriever()

# Retrieve NTSB incidents
incident_result = incident_retriever.retrieve(
    query="engine failure during takeoff",
    top_k=5
)

print(f"Retrieved {len(incident_result.documents)} NTSB incidents:")
for i, doc in enumerate(incident_result.documents, 1):
    print(f"\n{i}. Score: {doc.score:.3f}")
    print(f"   ID: {doc.id}")
    print(f"   Text: {doc.text[:200]}...")

In [ ]:
# Retrieve aviation news
news_result = news_retriever.retrieve(
    query="Boeing safety",
    top_k=3
)

print(f"Retrieved {len(news_result.documents)} news articles:")
for i, doc in enumerate(news_result.documents, 1):
    print(f"\n{i}. Score: {doc.score:.3f}")
    print(f"   Text: {doc.text[:200]}...")

## 7. Intent Classification and Strategy Selection

In [ ]:
from agentic_rag.orchestration import IntentClassifier, StrategySelector

classifier = IntentClassifier()
strategy_selector = StrategySelector()

# Test different question types
test_questions = [
    "What caused the crash of N12345?",
    "Was the pilot properly certified for the flight?",
    "What are the VFR weather minimums?",
    "Compare pilot fatigue rules with fatigue-related incidents"
]

for q in test_questions:
    intent, confidence, refinement = classifier.classify(q)
    plan = strategy_selector.select(intent, q)  # Method is 'select', not 'select_strategy'
    
    print(f"Q: {q}")
    print(f"   Intent: {intent.value} ({confidence})")
    print(f"   Strategy: {plan.strategy.value}")
    print(f"   Sources: {[s.value for s in plan.sources]}")
    print()

## 8. Context Refinement

In [ ]:
from agentic_rag.refinement import Deduplicator, Synthesizer

# Get some documents first
result = incident_retriever.retrieve("fuel exhaustion accidents", top_k=8)
print(f"Retrieved {len(result.documents)} documents")

# Deduplicate
deduplicator = Deduplicator()
dedup_result = deduplicator.deduplicate(
    documents=result.documents,
    query="fuel exhaustion accidents"
)

print(f"\nAfter deduplication:")
print(f"  Input: {dedup_result.input_count}")
print(f"  Output: {dedup_result.output_count}")
print(f"  Dropped: {len(dedup_result.dropped)}")

In [ ]:
# Synthesize (alternative refinement)
synthesizer = Synthesizer()
synth_result = synthesizer.synthesize(
    documents=result.documents,
    query="fuel exhaustion accidents"
)

print(f"After synthesis:")
print(f"  Input: {synth_result.input_count}")
print(f"  Output: {synth_result.output_count}")
print(f"\nSynthesized context:")
print(synth_result.documents[0].text[:500] + "...")

## 9. MLflow Run Example with Custom Experiment

In [ ]:
# Set a custom experiment for this analysis
custom_experiment = os.environ.get('MLFLOW_EXPERIMENT_NAME', 'agentic-rag-aviation')
mlflow.set_experiment(custom_experiment)

# End any existing runs first
if mlflow.active_run():
    mlflow.end_run()

# NOTE: agentic.query() creates its own MLflow run internally, so we don't wrap it
# Instead, we run the query and then log additional info in a separate run

# Execute query (this creates its own MLflow run)
response = await agentic.query(
    question="Cessna stall spin accidents",
    refinement_mode=RefinementMode.DEDUP,
    top_k=10,
    include_trace=True
)

# The query's MLflow run is already complete - let's show the results
print("Query complete!")
print(f"Experiment: {custom_experiment}")

# Display trace info
if response.trace:
    print(f"\nMLflow Run ID: {response.trace.mlflow_run_id}")
    
    # Get metrics from trace using correct attributes
    if response.trace.retrievals:
        total_docs = sum(r.get('document_count', 0) for r in response.trace.retrievals)
        print(f"Total documents retrieved: {total_docs}")
    
    if response.trace.refinement:
        print(f"Documents after refinement: {response.trace.refinement.get('output_count', 'N/A')}")
    
    if response.trace.total_duration_ms:
        print(f"Duration: {response.trace.total_duration_ms:.0f}ms")

print(f"\nAnswer summary: {response.answer.summary[:200]}...")

## 10. Batch Query Analysis

In [ ]:
# Run multiple queries for analysis
analysis_queries = [
    "What are common causes of general aviation accidents?",
    "Show me helicopter accidents in mountainous terrain",
    "What accidents involved student pilots?",
    "Find incidents with engine failure during cruise",
]

results = []
for query in analysis_queries:
    response = await agentic.query(
        question=query,
        refinement_mode=RefinementMode.DEDUP,
        top_k=10,
        include_trace=True
    )
    
    # Extract metrics from trace using correct attribute names
    trace = response.trace
    docs_retrieved = 0
    docs_refined = 0
    
    if trace:
        # Get total docs from retrievals
        if trace.retrievals:
            docs_retrieved = sum(r.get('document_count', 0) for r in trace.retrievals)
        # Get refined docs from refinement dict
        if trace.refinement:
            docs_refined = trace.refinement.get('output_count', 0)
    
    results.append({
        'query': query,
        'intent': trace.intent.value if trace and trace.intent else None,
        'docs_retrieved': docs_retrieved,
        'docs_refined': docs_refined,
        'duration_ms': trace.total_duration_ms if trace else 0,
        'summary_length': len(response.answer.summary) if response.answer.summary else 0
    })
    print(f"Processed: {query[:40]}...")

# Display results as table
import pandas as pd
df = pd.DataFrame(results)
print("\n" + "=" * 80)
print("BATCH ANALYSIS RESULTS")
print("=" * 80)
df

## Summary

This notebook demonstrated:

1. **Configuration** - Setting up API keys and MLflow via environment variables
2. **MLflow Tracing** - Automatic tracing of RAG pipeline execution
3. **Agentic RAG** - Running queries with intent classification and context refinement
4. **Comparison** - Comparing agentic vs traditional RAG approaches
5. **Low-level APIs** - Direct access to retrievers, classifiers, and refiners
6. **Batch Analysis** - Processing multiple queries for analysis

### Local Development URLs

| Service | URL |
|---------|-----|
| Agentic RAG API | http://localhost:8000 |
| Frontend UI | http://localhost:3000 |
| Qdrant Vector DB | http://localhost:6333 |
| MLflow Tracking | http://localhost:5001 |

### Environment Variables Reference

```bash
# Local Development Defaults
export QDRANT_URL=http://localhost:6333
export MLFLOW_TRACKING_URI=http://localhost:5001
export LLM_PROVIDER=anthropic
export MLFLOW_EXPERIMENT_NAME=agentic-rag-aviation

# Required API Keys
export ANTHROPIC_API_KEY=sk-ant-...   # If using Anthropic
export OPENAI_API_KEY=sk-proj-...     # If using OpenAI

# Start services
docker-compose up -d
```